# KumoRFM on SQL Databases

[[**Blog**](https://kumo.ai/company/news/kumo-relational-foundation-model/) | [**Paper**](https://kumo.ai/research/kumo_relational_foundation_model.pdf)]

**KumoRFM (Kumo Relational Foundation Model)** is a Foundation Model for machine learning on enterprise data. With just your data and a few lines of code, you can generate accurate predictions in real-time: no model training or pipelines required.

In [ ]:
!wget https://kumo-public-datasets.s3.us-west-2.amazonaws.com/sqlite/data.sqlite

In [ ]:
!pip install --pre kumoai[sqlite]

## Inspect Database

`KumoRFM` can connect to any SQL database or any in-memory data. For this purpose of the demo, we have prepared a `sqlite` database we can connect to:

In [ ]:
import sqlite3

with sqlite3.connect('data.sqlite') as conn:
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    print(f"Available Tables: {', '.join(tables)}")

    cursor.execute("SELECT user_id FROM users LIMIT 100")
    user_ids = [row[0] for row in cursor.fetchall()]
    print(f"Retrieved User IDs: {', '.join(str(user_id) for user_id in user_ids[:5])}, ...")

## Initialize KumoRFM

`KumoRFM` provides an [SDK](https://kumo-ai.github.io/kumo-sdk/docs/get_started/rfm/index.html) in Python.
The Kumo SDK is available for Python 3.10 to Python 3.14.

In [ ]:
from kumoai.experimental import rfm

You will need an API key to make calls to KumoRFM.

In [ ]:
rfm.authenticate()

In [ ]:
rfm.init()

## Connect KumoRFM to a Database

`KumoRFM` can directly connect to your database, and automatically infers relations and semantic types:

In [ ]:
graph = rfm.Graph.from_sqlite('data.sqlite')

In [ ]:
graph['users'].print_metadata()

In [ ]:
graph['orders'].print_metadata()

In [ ]:
graph.visualize(show_columns=False)

## Query KumoRFM

We are now ready to plug the graph into `KumoRFM` to make predictions!

The great thing about the graph is that it's a one-time setup—once it's in place, you can generate a variety of predictions from it and power many business use cases.

In [ ]:
model = rfm.KumoRFM(graph)

You can talk to `KumoRFM` via our **Predictive Query Language (PQL)**:

<div align="left">
  <img src="https://kumo-sdk-public.s3.us-west-2.amazonaws.com/rfm-colabs/predictive-query-structure.png" width="500" />
</div>

#### Predict Customer Churn

Predict the likelihood that users will place zero orders in the next 30 days:

In [ ]:
query = """
PREDICT COUNT(orders.*, 0, 30, days)=0
FOR EACH users.user_id
"""
model.predict(query, indices=user_ids)

Evaluate the model's predictions on historical data:

In [ ]:
model.evaluate(query)

Explain the predictions:

In [ ]:
model.predict(query, indices=user_ids[:1], explain=True)

#### Predict Next Best Items

Predict the top-10 items that users are likely going to buy in the next 7 days:

In [ ]:
query = """
PREDICT LIST_DISTINCT(orders.item_id, 0, 7, days) RANK TOP 10
FOR EACH users.user_id
"""
df = model.predict(query, indices=user_ids)
df

Visualizing recommended articles:

In [ ]:
from IPython.display import HTML, display

with sqlite3.connect('data.sqlite') as conn:
    cursor = conn.cursor()
    sql = f"""
    SELECT image_url FROM items
    WHERE item_id IN ({','.join(['?'] * len(df))})
    """
    cursor.execute(sql, df['CLASS'].tolist())
    urls = [row[0] for row in cursor.fetchall()]

html = "".join(
    f'<img src="{url}" style="width:150px; display:inline-block; margin-right:5px;">'
    for url in urls[:5]
)
display(HTML(html))

Evaluate the model's predictions on historical data:

In [ ]:
model.evaluate(query, metrics=['map@100'])

#### Predict Custom Tasks

Besides querying `KumoRFM` via our predictive query language, you can also bring in any custom task that cannot be formulated from the graph directly. This also allows you fine-grained control over in-context examples:

In [ ]:
import pandas as pd
import numpy as np

task = rfm.TaskTable(
    task_type='binary_classification',
    context_df=pd.DataFrame({
        'item_id': range(100),
        'target': np.random.randint(0, 2, size=100),
    }),
    pred_df=pd.DataFrame({
        'item_id': range(100, 110),
    }),
    entity_table_name='items',
    entity_column='item_id',
    target_column='target',
)
task

In [ ]:
model.predict_task(task)

<div align="left">
  <img src="https://kumo-sdk-public.s3.us-west-2.amazonaws.com/rfm-colabs/kumo_ai_logo.jpeg" width="30" />
</div>
